In [2]:
import sys
!{sys.executable} -m pip install nltk pandas scikit-learn numpy

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)

   ------------- -------------------------- 1/3 [regex]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   -------------------------- ------------- 2/3 [nltk]
   ---

In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Acer\AppData\Roaming\nltk_data...


True

In [4]:
df_resumes = pd.read_csv('data/UpdatedResumeDataSet.csv')
df_jobs = pd.read_csv('data/job_title_des.csv')

df_jobs.drop(columns=['Unnamed: 0'], inplace=True)

print("Resumes:", df_resumes.shape)
print("Jobs:", df_jobs.shape)

Resumes: (962, 2)
Jobs: (2277, 2)


In [8]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return ' '.join(tokens)

df_resumes['cleaned'] = df_resumes['Resume'].apply(clean_text)
df_jobs['cleaned'] = df_jobs['Job Description'].apply(clean_text)

print("✅ Cleaning done!")
print(df_resumes.columns.tolist())
print(df_jobs.columns.tolist())

✅ Cleaning done!
['Category', 'Resume', 'cleaned']
['Job Title', 'Job Description', 'cleaned']


In [7]:
print(df_resumes.columns.tolist())
print(df_jobs.columns.tolist())

['Category', 'Resume']
['Job Title', 'Job Description']


In [5]:
df_resumes.to_csv('data/cleaned_resumes.csv', index=False)
df_jobs.to_csv('data/cleaned_jobs.csv', index=False)

print("✅ Saved cleaned_resumes.csv")
print("✅ Saved cleaned_jobs.csv")

✅ Saved cleaned_resumes.csv
✅ Saved cleaned_jobs.csv


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

# Fit on both resumes and job descriptions combined
all_text = list(df_resumes['cleaned']) + list(df_jobs['cleaned'])
tfidf.fit(all_text)

# Transform each separately
resume_vectors = tfidf.transform(df_resumes['cleaned'])
job_vectors = tfidf.transform(df_jobs['cleaned'])

print("✅ TF-IDF done!")
print("Resume matrix shape:", resume_vectors.shape)
print("Job matrix shape:", job_vectors.shape)

✅ TF-IDF done!
Resume matrix shape: (962, 5000)
Job matrix shape: (2277, 5000)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def match_resume_to_job(resume_text, job_index):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    job_vec = job_vectors[job_index]
    
    score = cosine_similarity(resume_vec, job_vec)[0][0]
    return round(score * 100, 2)

# Test it
sample_resume = df_resumes['Resume'][0]
sample_job = df_jobs['Job Title'][0]
score = match_resume_to_job(sample_resume, 0)

print(f"Resume Category: {df_resumes['Category'][0]}")
print(f"Job Title: {sample_job}")
print(f"Match Score: {score}%")

Resume Category: Data Science
Job Title: Flutter Developer
Match Score: 0.53%


In [11]:
def get_top_matches(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    
    scores = cosine_similarity(resume_vec, job_vectors)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        results.append({
            'job_title': df_jobs['Job Title'][i],
            'score': round(scores[i] * 100, 2)
        })
    
    return results

# Test it
sample_resume = df_resumes['Resume'][0]
top_matches = get_top_matches(sample_resume)

print(f"Resume Category: {df_resumes['Category'][0]}")
print("\nTop 5 Matching Jobs:")
for match in top_matches:
    print(f"  {match['job_title']} — {match['score']}%")

Resume Category: Data Science

Top 5 Matching Jobs:
  Machine Learning — 26.96%
  Machine Learning — 25.45%
  Machine Learning — 24.21%
  Machine Learning — 24.08%
  Machine Learning — 23.19%


In [12]:
# Drop duplicate job titles, keep first occurrence
df_jobs_unique = df_jobs.drop_duplicates(subset='Job Title').reset_index(drop=True)

# Re-vectorize with deduplicated jobs
job_vectors_unique = tfidf.transform(df_jobs_unique['cleaned'])

def get_top_matches_unique(resume_text, top_n=5):
    cleaned_resume = clean_text(resume_text)
    resume_vec = tfidf.transform([cleaned_resume])
    
    scores = cosine_similarity(resume_vec, job_vectors_unique)[0]
    top_indices = scores.argsort()[::-1][:top_n]
    
    results = []
    for i in top_indices:
        results.append({
            'job_title': df_jobs_unique['Job Title'][i],
            'score': round(scores[i] * 100, 2)
        })
    
    return results

# Test again
top_matches = get_top_matches_unique(df_resumes['Resume'][0])

print(f"Resume Category: {df_resumes['Category'][0]}")
print("\nTop 5 Matching Jobs:")
for match in top_matches:
    print(f"  {match['job_title']} — {match['score']}%")

Resume Category: Data Science

Top 5 Matching Jobs:
  Machine Learning — 15.15%
  Software Engineer — 9.25%
  Java Developer — 8.02%
  Full Stack Developer — 7.0%
  Backend Developer — 6.62%


In [13]:
import pickle

# Save tfidf vectorizer
with open('models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save job vectors
with open('models/job_vectors.pkl', 'wb') as f:
    pickle.dump(job_vectors_unique, f)

# Save deduplicated jobs dataframe
df_jobs_unique.to_csv('data/cleaned_jobs_unique.csv', index=False)

print("✅ tfidf_vectorizer.pkl saved")
print("✅ job_vectors.pkl saved")
print("✅ cleaned_jobs_unique.csv saved")

✅ tfidf_vectorizer.pkl saved
✅ job_vectors.pkl saved
✅ cleaned_jobs_unique.csv saved
